# Lab 3a — Local Post-hoc Explainability on Structured Data
**Teaching assistant**: Eleonora Poeta (eleonora.poeta@polito.it)

---

This lab covers **LIME (Local Interpretable Model-agnostic Explanations)**, a local post-hoc explainability technique applied to a black-box classifier on the Titanic dataset.

| Concept | Description |
|---|---|
| **Local surrogate** | An interpretable model that approximates the black-box *near a specific instance* |
| **Perturbation** | Slightly modified copies of the instance used to probe the black-box |
| **Proximity weighting** | Perturbed samples closer to the instance of interest are given higher weight |

**Guide:**
- ⚙️ Data preprocessing (Titanic + LIME-specific encoding)
- 🔍 LIME explainer setup
- 🎲 Exploring LIME parameters


## 🔍 What is LIME?

LIME is a **local surrogate model**. It tests **what happens to the predictions** when you give **variations of your data** into the machine learning model.

The main steps are:

1. LIME generates **a new dataset** consisting of **perturbed samples** and the corresponding **predictions** of the black-box model.
2. On this new dataset → LIME trains an **interpretable model** (weighted by the proximity of the sampled instances to the instance of interest).
3. The learned model should be a **good local approximation** of the machine learning model predictions, but it does **not** have to be a good global approximation.

> 💡 The key insight: we only need the **prediction function** of the black-box — not its internals.


## ⚙️ Data Preprocessing (~15 min)

We use the **[Titanic](https://www.openml.org/search?type=data&sort=runs&id=40945&status=active)** dataset loaded from OpenML.
The preprocessing pipeline for LIME is slightly different from previous labs: **categorical features are Label-Encoded first** (for the explainer), then additionally One-Hot Encoded (for the classifier).

> 💡 Apply the same good habits from Lab 2: always fit transformations on the **training set only**, then apply them to the test set!


### ⚙️ Step 1 — Imports & Load

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

import pandas as pd
import numpy as np

# Import the required libraries for this exercise
### Write your code here!

print("✅ Libraries loaded")

### ⚙️ Step 2 — Load the Titanic Dataset

The [Titanic dataset](https://www.openml.org/search?type=data&sort=runs&id=40945&status=active) contains passenger information and whether they survived.
We want to predict the column **`survived`** (0 = no, 1 = yes).


In [ ]:
# Load input features and target variable
df, y = fetch_openml("titanic", version=1, as_frame=True, parser='auto', return_X_y=True)

# The "survived" column contains the target variable
df["survived"] = y

### ⚙️ Step 3 — Stratified Train / Test Split

> 🔑 **Key reminder:** split *before* any transformation. Fitting scalers or imputers on the full dataset causes **data leakage**.
Use an **80/20** ratio, **shuffle** the dataset, and **stratify** by the target variable.


In [ ]:
#### START CODE HERE ####
# Split 80% train / 20% test, shuffle=True, stratify by 'survived'
df_train, df_test = train_test_split(df, test_size=0.2, shuffle=True, random_state=42, stratify=df['survived'])

#### END CODE HERE ####

### ⚙️ Step 4 — Handle Missing Values

> 🔑 **Key reminder:** compute statistics (mean, median, mode) on the **training set only**, then apply to both sets.

Fill missing values as follows:
- `age` → training **mean**
- `fare` → training **median**
- `embarked` → training **most frequent** value


In [ ]:
#### START CODE HERE ####
# Fill 'age' NaN with the TRAINING mean
print(f'Number of null values in Train before pre-processing: {df_train.age.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test before pre-processing: {df_test.age.isnull().sum()}/{len(df_test)}')

df_train['age'] = df_train['age'].fillna(df_train['age'].mean())
df_test['age'] = df_test['age'].fillna(df_train['age'].mean())

print(f'Number of null values in Train after pre-processing: {df_train.age.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test after pre-processing: {df_test.age.isnull().sum()}/{len(df_test)}')

# Fill 'fare' NaN with the TRAINING median
print(f'Number of null values in Train before pre-processing: {df_train.fare.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test before pre-processing: {df_test.fare.isnull().sum()}/{len(df_test)}')

df_train['fare'] = df_train['fare'].fillna(df_train['fare'].median())
df_test['fare'] = df_test['fare'].fillna(df_train['fare'].median())

print(f'Number of null values in Train after pre-processing: {df_train.fare.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test after pre-processing: {df_test.fare.isnull().sum()}/{len(df_test)}')

# Fill 'embarked' NaN with the TRAINING most frequent value
print(f'Number of null values in Train before pre-processing: {df_train.embarked.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test before pre-processing: {df_test.embarked.isnull().sum()}/{len(df_test)}')

imp = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
df_train[['embarked']] = imp.fit_transform(df_train[['embarked']])
df_test[['embarked']] = imp.transform(df_test[['embarked']])

print(f'Number of null values in Train after pre-processing: {df_train.embarked.isnull().sum()}/{len(df_train)}')
print(f'Number of null values in Test after pre-processing: {df_test.embarked.isnull().sum()}/{len(df_test)}')

#### END CODE HERE ####

### ⚙️ Step 5 — Drop Uninformative & Leaky Columns

Remove columns that are either **not informative** for the task, or that **leak information** about the target variable (`survived`):
- Not informative: `name`, `ticket`
- Leaky (contain post-hoc survival info): `cabin`, `body`, `boat`, `home.dest`


In [ ]:
#### START CODE HERE ####
# Drop: 'name', 'ticket', 'cabin', 'body', 'boat', 'home.dest'

df_train = df_train.drop(columns=['name','ticket','cabin', 'body', 'boat', 'home.dest'])
df_test = df_test.drop(columns=['name','ticket','cabin', 'body', 'boat', 'home.dest'])
#### END CODE HERE ####

### ⚙️ Step 6 — Extract Features and Target

Separate the input features `X` and the target variable `y` for both the training and test sets.


In [ ]:
#### START CODE HERE ####
# Extract X_train, X_test, y_train, y_test

y_train = df_train['survived']               # Target variable training set
X_train = df_train.drop('survived', axis=1)  # Features training set

# Extract target variable and input features for the testing data
y_test = df_test['survived']                 # Target variable test set
X_test = df_test.drop('survived', axis=1)    # Features test set

#### END CODE HERE ####

### ⚙️ Step 7 — LIME-Specific Encoding

The LIME explainer (and most classifiers) requires **numerical data**, even for categorical features.
We apply a **two-stage encoding** strategy:

| Stage | Encoder | Used by |
|---|---|---|
| **Stage 1** | `LabelEncoder` per categorical column | LIME explainer |
| **Stage 2** | `OneHotEncoder` + `MinMaxScaler` via `ColumnTransformer` | Random Forest classifier |

> 🔑 **Key reason:** the explainer must ensure a categorical feature only takes **one value** at a time. LabelEncoding achieves this, while OHE would split it across multiple binary columns.

We also save a `categorical_names` dictionary mapping each feature index to the list of its original string labels — this allows LIME to display readable feature values in the explanation.


#### Step 7a — Identify Categorical Columns

Identify the **categorical columns** (dtype `category` or `object`) and save their **column indices** — LIME requires indices, not names.
Both `category` and `object` dtypes represent categorical columns here.


In [ ]:
# Display .info() to inspect dtypes
### Write your code here!

In [ ]:
#### START CODE HERE ####
# Identify categorical columns and save their indices
# categorical_cols should be a list of integer indices


#### END CODE HERE ####

#### Step 7b — LabelEncode for LIME

For each categorical feature:
1. Instantiate a `LabelEncoder()` and **fit+transform** on the **training set**.
2. Save the mapping: `categorical_names[feature_index] = le.classes_`
3. Save the encoder: `le_dict[feature_index] = le`
4. For the **test set**, apply only `.transform()` using the saved encoder.


In [ ]:
categorical_names = {}
le_dict = {}

#### START CODE HERE ####
# Fit LabelEncoder on train, transform train; then transform test
for feature in categorical_cols:
    ## Continue with your code here
    pass

#### END CODE HERE ####

print("categorical_names:", categorical_names)

In [ ]:
categorical_names_test = {}

#### START CODE HERE ####
# Build categorical_names_test for the test set
for feature in categorical_cols:
    ## Continue with your code here
    pass

#### END CODE HERE ####

#### Step 7c — OneHotEncoder + MinMaxScaler for the Classifier

The **classifier** needs proper encoding — OHE avoids treating categorical codes as continuous values.

> ⚠️ We use this encoder **only for the classifier**, not for the explainer.

1. Instantiate a `OneHotEncoder()` for categorical columns.
2. Instantiate a `MinMaxScaler()` for numerical columns.
3. Combine with a `ColumnTransformer()`.
4. Fit on training data, apply to both train and test.


In [ ]:
#### START CODE HERE ####
# Identify numerical column indices

# Initialize OneHotEncoder

# Initialize MinMaxScaler

# Create and fit ColumnTransformer

# Apply to train and test

#### END CODE HERE ####

> ✅ **Preprocessing complete.** Remember the LIME-specific rules:
> 1. Split *before* fitting any transformer.
> 2. LabelEncode for the **explainer**, OHE+scale for the **classifier**.
> 3. Always fit encoders on **train only**, then apply to both sets.


## Exercise 1 — Train the Black-Box Classifier 🌲

We train a **RandomForestClassifier** as our black-box model.

**Your tasks:**
1. Fit a `RandomForestClassifier` with `n_estimators=500`.
2. Calculate predictions with `.predict()`.
3. Calculate the `accuracy_score()`.

---

### 1.1 — Fit & Evaluate


In [ ]:
#### START CODE HERE ####
# Fit RandomForestClassifier with n_estimators=500


#### END CODE HERE ####

In [ ]:
#### START CODE HERE ####
# Calculate y_pred with .predict()


#### END CODE HERE ####

In [ ]:
#### START CODE HERE ####
# Calculate and print accuracy_score


#### END CODE HERE ####

## Exercise 2 — Explaining Predictions with LIME 🔍

Now we explain individual predictions using **LIME**. Before starting, install the library and import the tabular module:

```python
!pip install lime
from lime import lime_tabular
```

**Key steps:**
1. Fix the random seed with `np.random.seed(42)`.
2. Instantiate the explainer: `lime_tabular.LimeTabularExplainer` — read the [documentation](https://lime-ml.readthedocs.io/en/latest/lime.html#module-lime.lime_tabular) to understand each parameter.
3. Define a **custom prediction function** `predict_fn` that applies the `ColumnTransformer` before calling the classifier (LIME feeds raw un-OHE'd data, so we must transform it on-the-fly).
4. Use `explainer.explain_instance` to explain instance `i=0`.

> 💡 *What can you infer? What is the predicted class for that instance?*

---

### 2.1 — Install & Import


In [ ]:
!pip install lime
from lime import lime_tabular

### 2.2 — Fix Random Seed

In [ ]:
#### START CODE HERE ####
# Fix the random seed with np.random.seed(42)


#### END CODE HERE ####

### 2.3 — Instantiate the LimeTabularExplainer

In [ ]:
explainer = lime_tabular.LimeTabularExplainer(
    X_train.values,                    # LIME requires a numpy array
    mode='classification',
    class_names=['not survived', 'survived'],
    feature_names=X_train.columns,
    categorical_features=categorical_cols,
    categorical_names=categorical_names,
    kernel_width=3,
    verbose=True
)

### 2.4 — Define the Custom Prediction Function

In [ ]:
def predict_fn(x):
    # LIME passes raw (Label-Encoded) data, so we must apply the ColumnTransformer first
    temporary_df = pd.DataFrame(x, columns=X_train.columns, dtype='object')
    print(temporary_df.head(2))
    transf = ct.transform(temporary_df)
    pred = rf.predict_proba(transf).astype(float)
    return pred

### 2.5 — Explain Instance `i=0`

In [ ]:
#### START CODE HERE ####
i = 0
exp = explainer.explain_instance(
    X_test.values[i],
    predict_fn,
    num_samples=3
)
exp.show_in_notebook()
#### END CODE HERE ####

## Exercise 3 — Exploring LIME Parameters 🎲

Now it's time to **play with LIME** and understand how each parameter affects the explanation.

**Your tasks:**
1. Instantiate **a new `LimeTabularExplainer`** (same config as before).
2. Use the **same `predict_fn`**.
3. Call `explain_instance` for **instance `i=1`**, run it **5 times**.
   > *Did you always get the same explanation? If not — what is the missing step?*
4. Change `num_samples=15`. *What is the role of this parameter?*
5. Vary `num_features` between 1 and 6. *Where do you see a change?*
6. Change `distance_metric='l2'`. *Where is the distance metric used in LIME?*

---

### 3.1 — Repeated Explanations (Stability)


In [ ]:
#### START CODE HERE ####
# Instantiate a new LimeTabularExplainer and run explain_instance 5 times for i=1


#### END CODE HERE ####

### 3.2 — Effect of `num_samples`

In [ ]:
#### START CODE HERE ####
# Try num_samples=15 and observe the effect


#### END CODE HERE ####

### 3.3 — Effect of `num_features`

In [ ]:
#### START CODE HERE ####
# Vary num_features from 1 to 6 and observe changes in the explanation


#### END CODE HERE ####

### 3.4 — Effect of `distance_metric`

In [ ]:
#### START CODE HERE ####
# Try distance_metric='l2' and observe where it makes a difference


#### END CODE HERE ####

### 🔍 Reflection Questions

1. **Stability:** Why did running LIME multiple times without fixing the seed produce different explanations? What does this tell you about the reliability of local explanations?

2. **num_samples:** How does increasing `num_samples` affect the quality of the local surrogate model? What is the trade-off with computation time?

3. **LabelEncoder vs OneHotEncoder:** Why does the LIME explainer work on LabelEncoded data while the classifier uses OHE data? What would go wrong if you passed OHE data directly to the explainer?

4. **Local vs Global:** LIME provides a *local* explanation. Can it mislead you about the model's *global* behaviour? Give an example using the Titanic context (e.g. a first-class woman vs a third-class man).
